In [16]:
import re
import pandas as pd

In [23]:
arquivo = r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\completov4\io_paths.txt"
worst_path = r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\completov4\worst_path.txt"

In [ ]:

def max_ff_from_file(filename):
    max_ff = float('-inf')

    with open(filename, 'r') as f:
        for line in f:
            # extrai todos os números da linha
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", line)

            # esperamos pelo menos 4 valores (RR RF FR FF)
            if len(nums) >= 4:
                ff = float(nums[-1])  # último número é FF
                max_ff = max(max_ff, ff)

    return max_ff


print("Maior FF =", max_ff_from_file("file1.txt"))

Maior FF = 37.0


In [25]:
def max_ff_with_ports(filename):
    max_ff = float('-inf')
    max_input = None
    max_output = None

    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()

            # só linhas de dados
            if line.startswith('; a'):
                parts = [p.strip() for p in line.split(';') if p.strip()]

                try:
                    input_port  = parts[0]
                    output_port = parts[1]
                    ff_value    = float(parts[5])  # FF

                    if ff_value > max_ff:
                        max_ff = ff_value
                        max_input = input_port
                        max_output = output_port

                except:
                    pass

    return max_input, max_output, max_ff


# uso
inp, outp, ff = max_ff_with_ports(worst_path)

print(f"Maior FF = {ff}")
print(f"Entrada  = {inp}")
print(f"Saída    = {outp}")

print(f"report_path -from [get_keepers {inp}] -to [get_keepers {outp}] -npaths 1 -panel_name {{Report Path}} -multi_corner")

Maior FF = -inf
Entrada  = None
Saída    = None
report_path -from [get_keepers None] -to [get_keepers None] -npaths 1 -panel_name {Report Path} -multi_corner


In [32]:
def max_delay_with_ports(filename):
    max_delay = float("-inf")
    max_input = None
    max_output = None

    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            # linha da tabela Summary:
            # ; 14.641 ; a9[4] ; sum[15] ;
            if line.startswith(";"):
                parts = [p.strip() for p in line.split(";") if p.strip()]

                if len(parts) == 3:
                    try:
                        delay = float(parts[0])
                        input_port = parts[1]
                        output_port = parts[2]

                        if delay > max_delay:
                            max_delay = delay
                            max_input = input_port
                            max_output = output_port
                    except ValueError:
                        pass

    return max_input, max_output, max_delay


# uso
worst_path = r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\completov4\worst_path.txt"

inp, outp, delay = max_delay_with_ports(worst_path)

print(f"Maior Delay = {delay} ns")
print(f"Entrada     = {inp}")
print(f"Saída       = {outp}")

print(
    f"report_path -from [get_keepers {{{inp}}}] "
    f"-to [get_keepers {{{outp}}}] "
    f"-npaths 1 -panel_name {{Report Path}} -multi_corner"
)

Maior Delay = 16.653 ns
Entrada     = a1[15]
Saída       = sum[17]
report_path -from [get_keepers {a1[15]}] -to [get_keepers {sum[17]}] -npaths 1 -panel_name {Report Path} -multi_corner


In [33]:
arquivo = r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\completov4\io_paths.txt"

paths = []
current = None

with open(arquivo, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        m = re.search(r"Path #(\d+): Delay is\s+([-+]?\d*\.\d+|\d+)", line)
        if m:
            if current:
                paths.append(current)
            current = {
                "path": int(m.group(1)),
                "delay": float(m.group(2)),
                "cell_total": None,
                "ic_total": None,
                "from": None,
                "to": None,
            }
            continue

        if current is None:
            continue

        m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
        if m:
            current["from"] = m.group(1).strip()

        m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
        if m:
            current["to"] = m.group(1).strip()

        # Linha tipo:
        # ; IC   ; ; 19 ; 7.013 ; 53 ; 0.000 ; 2.467 ;
        m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([-+]?\d*\.\d+|\d+)\s*;", line)
        if m:
            current["ic_total"] = float(m.group(1))

        # Linha tipo:
        # ; Cell ; ; 19 ; 6.301 ; 47 ; 0.000 ; 1.412 ;
        m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([-+]?\d*\.\d+|\d+)\s*;", line)
        if m:
            current["cell_total"] = float(m.group(1))

if current:
    paths.append(current)

df = pd.DataFrame(paths)

print("Quantidade de paths:", len(df))

print("\nMédias gerais:")
print("Delay médio:", df["delay"].mean(), "ns")
print("CELL total médio:", df["cell_total"].mean(), "ns")
print("IC total médio:", df["ic_total"].mean(), "ns")

print("\nMaiores valores:")
print("Maior Delay:")
print(df.loc[df["delay"].idxmax()])

print("\nMaior CELL total:")
print(df.loc[df["cell_total"].idxmax()])

print("\nMaior IC total:")
print(df.loc[df["ic_total"].idxmax()])

# df.to_csv("resumo_paths.csv", index=False)
# print("\nCSV salvo em resumo_paths.csv")

Quantidade de paths: 10000

Médias gerais:
Delay médio: 15.2448441 ns
CELL total médio: 7.009118299999999 ns
IC total médio: 8.2357258 ns

Maiores valores:
Maior Delay:
path                1
delay          16.653
cell_total       7.99
ic_total        8.663
from           a1[15]
to            sum[17]
Name: 0, dtype: object

Maior CELL total:
path                7
delay          16.593
cell_total      8.114
ic_total        8.479
from           a1[15]
to            sum[16]
Name: 6, dtype: object

Maior IC total:
path               28
delay          16.471
cell_total      7.163
ic_total        9.308
from           a2[12]
to            sum[17]
Name: 27, dtype: object
